# 알약 검출 베이스라인 — Google Colab

**사전 준비 (로컬 PC에서):**
1. Google Drive에 `팀 프로젝트/` 폴더 전체를 업로드
2. 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택
3. 위에서부터 순서대로 셀 실행

## 세션이 끊겼으면 셀 1번 부터 
세션 유지중이면 1번 부터 

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정
> Drive에 올린 폴더 경로에 맞게 `PROJECT_DIR` 을 수정하세요.

In [ ]:
import os

# Google Drive 내 프로젝트 루트 경로
PROJECT_DIR = '/content/drive/MyDrive/베이스라인_코랩버전v1.2'
BASELINE_DIR = os.path.join(PROJECT_DIR, 'baseline')
DATA_DIR = os.path.join(PROJECT_DIR, 'sprint_ai_project1_data')

print('PROJECT_DIR :', PROJECT_DIR)
print('BASELINE_DIR:', BASELINE_DIR)
print('DATA_DIR    :', DATA_DIR)

# 경로 존재 확인
for p in [PROJECT_DIR, BASELINE_DIR, DATA_DIR]:
    print(f'  {p} → exists={os.path.exists(p)}')

## 3. Drive 데이터를 Colab 로컬(/content)로 복사
> Drive 직접 읽기는 느리므로 한 번 복사하면 학습이 훨씬 빠릅니다.

In [ ]:
# import shutil

# LOCAL_DIR = '/content/pill_project'
# os.makedirs(LOCAL_DIR, exist_ok=True)

# # baseline 코드 복사
# if not os.path.exists(os.path.join(LOCAL_DIR, 'baseline')):
#     print('baseline 코드 복사 중...')
#     shutil.copytree(BASELINE_DIR, os.path.join(LOCAL_DIR, 'baseline'))
#     print('완료')
# else:
#     print('baseline 이미 존재, 스킵')

# # 데이터 복사
# LOCAL_DATA = os.path.join(LOCAL_DIR, 'data')
# if not os.path.exists(LOCAL_DATA):
#     print('데이터 복사 중... (시간이 걸릴 수 있습니다)')
#     shutil.copytree(DATA_DIR, LOCAL_DATA)
#     print('완료')
# else:
#     print('데이터 이미 존재, 스킵')

# # 이후 작업 디렉토리를 baseline으로 설정
# os.chdir(os.path.join(LOCAL_DIR, 'baseline'))
# print('현재 디렉토리:', os.getcwd())

In [ ]:
import os

os.chdir('/content/drive/MyDrive/베이스라인_코랩버전v1.2')
print('현재 디렉토리:', os.getcwd())
print('내용:', os.listdir('.'))

## 4. config 경로 업데이트
> Colab 로컬 경로로 data_root를 자동 수정합니다.

In [ ]:
import yaml

cfg_path = 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# cfg['data']['data_root'] = '/content/pill_project/data'
cfg['data']['data_root'] = '/content/drive/MyDrive/베이스라인_코랩버전/sprint_ai_project1_data'
cfg['data']['processed_dir'] = './data/processed'
cfg['data']['num_workers'] = 2  # Colab은 2가 안정적

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('config 업데이트 완료')
print('data_root:', cfg['data']['data_root'])

## 5. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt

## 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 7. 어노테이션 전처리 (최초 1회만 실행)

In [ ]:
!python scripts/preprocess.py

## 8. (선택) GT 박스 시각화

In [ ]:
!python scripts/visualize.py --n 3 --output outputs/vis

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

vis_files = sorted(glob.glob('outputs/vis/*.jpg'))[:3]
fig, axes = plt.subplots(1, len(vis_files), figsize=(15, 6))
for ax, path in zip(axes, vis_files):
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=8)
plt.tight_layout()
plt.show()

## 9. 학습

In [ ]:
!python train.py

## 10. 테스트 추론

In [ ]:
!python inference.py --checkpoint outputs/checkpoints/best.pth

## 11. 결과를 Google Drive에 백업

In [ ]:
import shutil

BACKUP_DIR = os.path.join(PROJECT_DIR, 'baseline/outputs')
os.makedirs(BACKUP_DIR, exist_ok=True)

# checkpoints 백업
shutil.copytree(
    'outputs/checkpoints',
    os.path.join(BACKUP_DIR, 'checkpoints'),
    dirs_exist_ok=True,
)
# predictions 백업
shutil.copytree(
    'outputs/predictions',
    os.path.join(BACKUP_DIR, 'predictions'),
    dirs_exist_ok=True,
)
print('Drive 백업 완료:', BACKUP_DIR)